# 🧬 Práctica: análisis de datos de nanoporo en la nube
**Curso de Análisis Bioinformático de Datos Genómicos aplicado al Diagnóstico Genético · Bloque II (Long-Read Sequencing)**
Belén de la Morena Barrio · CIBERER / Longseq Applications

---

### ¿Qué vas a hacer?
Partirás de las **lecturas crudas (FASTQ)** de un paciente secuenciado con nanoporo y recorrerás el pipeline completo que hemos visto en clase:

| Paso | Herramienta | Fichero que obtienes |
|---|---|---|
| 1. Control de calidad | Python | gráficas + métricas (N50, Q media) |
| 2. Alineamiento al genoma de referencia | **minimap2** + samtools | `.bam` + `.bai` |
| 3. Calidad y cobertura en la región (SERPINC1) | samtools | perfil de cobertura |
| 4. Llamada de variantes estructurales | **Sniffles2** | `.vcf` |
| 5. Metilación por alelo | **Methylartist** | gráfico `.png` |
| 6. Visualización | **IGV** | ¡tus propios ojos! 👀 |

### Cómo se usa este cuaderno
* **No tienes que instalar nada.** Todo corre en un ordenador en la nube, gratis.
* Ejecuta las celdas **en orden, de arriba abajo**: pulsa ▶ a la izquierda de cada celda o `Mayús + Intro`.
* **En GitHub Codespaces** (recomendado): la primera vez, arriba a la derecha pulsa **«Select Kernel» → Python Environments → Python 3**.
* **En Google Colab** (alternativa): haz primero **`Archivo → Guardar una copia en Drive`**.
* Los datos (FASTQ) ya están en la carpeta **`datos/`** del repositorio y los resultados se guardan en **`resultados/`**.
* ¿Prefieres la terminal? Abre una (menú ☰ → Terminal → New Terminal) y ejecuta `bash analisis_nanoporo.sh`: hace lo mismo paso a paso.

> 🎯 **El reto:** este paciente tiene un diagnóstico clínico claro, pero Sanger, NGS y MLPA salieron **negativos**. Tu misión es encontrar la causa genética.

## 0. Parámetros del ejercicio
No hace falta tocarlos (la profesora ya los ha rellenado), pero fíjate en qué significa cada uno.

In [ ]:
#@title Parámetros { display-mode: "form" }
# FASTQ del paciente dentro del repositorio (sin comprimir .fastq o comprimido .fastq.gz)
FASTQ_REPO = "datos/P1_SERPINC1.fastq"  #@param {type:"string"}
# Repositorio de GitHub (usuario/repositorio). Solo se usa en Google Colab
REPO_GITHUB = "TU_USUARIO/practica-nanoporo"  #@param {type:"string"}
MUESTRA = "paciente1"  #@param {type:"string"}
# Cromosoma donde está la región enriquecida (adaptive sampling)
CROMOSOMA = "chr1"  #@param {type:"string"}
# Región / gen de interés para inspeccionar (coordenadas hg38)
REGION = "chr1:173895036-173926091"  #@param {type:"string"}
GEN = "SERPINC1"  #@param {type:"string"}
# La metilación (etiquetas MM/ML de Dorado) se detecta automáticamente en el paso 2

import os, sys
EN_COLAB = "google.colab" in sys.modules
import glob
from pathlib import Path

def carpeta_del_repositorio():
    """Busca la carpeta que contiene 'datos/' (aunque se re-ejecute esta celda)."""
    candidatas = [os.environ.get("CODESPACE_VSCODE_FOLDER"), os.getcwd()]
    candidatas += [str(p) for p in Path(os.getcwd()).parents]
    candidatas += sorted(glob.glob("/workspaces/*"))
    for c in candidatas:
        if c and os.path.isdir(os.path.join(c, "datos")):
            return c
    return os.environ.get("CODESPACE_VSCODE_FOLDER") or os.getcwd()

RAIZ = "/content" if EN_COLAB else carpeta_del_repositorio()
TRABAJO = os.path.join(RAIZ, "resultados")
os.makedirs(TRABAJO, exist_ok=True)
os.chdir(TRABAJO)
print("Entorno:", "Google Colab" if EN_COLAB else "GitHub Codespaces / Jupyter")
print("Trabajando en", TRABAJO)

# --- Función para VER lo que se ejecuta (comando + explicación + salida en directo) ---
import subprocess
def ejecuta(comando, explicacion=None):
    print("\033[1;34m" + "─" * 80 + "\033[0m")
    if explicacion:
        for parte, significado in explicacion:
            print(f"  \033[1m{parte:<28}\033[0m {significado}")
        print()
    print("\033[1;32m$ " + comando + "\033[0m\n")
    proc = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, executable="/bin/bash")
    for linea in proc.stdout:
        print("   " + linea, end="")
    proc.wait()
    print(("\n✅ Terminado" if proc.returncode == 0 else f"\n❌ Error (código {proc.returncode})"))


## 1. Instalación de las herramientas (~1 min)
Instalamos **minimap2** (alineador), **samtools** (manejo de BAM), **Sniffles2** (variantes estructurales), **Methylartist** (metilación) e **igv-notebook** (IGV dentro del cuaderno).

In [ ]:
import shutil
if EN_COLAB:
    !apt-get -qq update
    !apt-get -qq install -y minimap2 samtools tabix
    !pip -q install sniffles==2.4 methylartist igv-notebook
else:  # Codespaces: normalmente ya vienen instaladas; instalamos solo lo que falte
    if not shutil.which("minimap2"):
        !sudo apt-get -qq update
        !sudo apt-get -qq install -y minimap2 samtools tabix zip
    if not shutil.which("sniffles"):
        !pip -q install sniffles==2.4 pandas matplotlib
    if not shutil.which("methylartist"):
        !pip -q install methylartist
print('✅ Herramientas listas')

In [ ]:
!minimap2 --version && samtools --version | head -1 && sniffles --version 2>/dev/null | tail -1 && methylartist --version 2>/dev/null | tail -1

## 2. Datos de partida (~2 min)
* **FASTQ** del paciente: está en la carpeta `datos/` del repositorio.
* **Genoma de referencia**: solo el cromosoma de interés del hg38 (para ir más rápido), descargado de UCSC.

In [ ]:
import urllib.request

# 1) FASTQ: lo buscamos en el repositorio (Codespaces) o lo descargamos de GitHub (Colab)
origen = os.path.join(RAIZ, FASTQ_REPO)
if not os.path.exists(origen) and EN_COLAB:
    url = f"https://raw.githubusercontent.com/{REPO_GITHUB}/main/{FASTQ_REPO}"
    print("Descargando", url)
    os.makedirs(os.path.dirname(origen), exist_ok=True)
    urllib.request.urlretrieve(url, origen)
if not os.path.exists(origen):
    # ¿Está con otro nombre o en otra carpeta? Lo buscamos
    nombre = os.path.basename(FASTQ_REPO)
    parecidos = [f for f in glob.glob(os.path.join(RAIZ, "**", "*"), recursive=True)
                 if os.path.isfile(f) and nombre.split(".")[0].lower() in os.path.basename(f).lower()
                 and "/resultados/" not in f]
    if len(parecidos) == 1:
        origen = parecidos[0]
        print(f"⚠️ No estaba en '{FASTQ_REPO}', pero lo he encontrado en: {origen}")
    else:
        carpeta = os.path.join(RAIZ, "datos")
        contenido = os.listdir(carpeta) if os.path.isdir(carpeta) else "(no existe la carpeta datos/)"
        raise SystemExit(f"\n❌ No encuentro el FASTQ en '{FASTQ_REPO}'.\n"
                         f"   Carpeta del repositorio: {RAIZ}\n"
                         f"   Lo que hay en datos/: {contenido}\n"
                         "   · Si lo subiste desde la web de GitHub DESPUÉS de crear el Codespace, tu Codespace aún no lo tiene:\n"
                         "     abre una terminal y escribe:  git pull\n"
                         "   · Si el nombre es distinto, cópialo exactamente en FASTQ_REPO (mayúsculas, .fastq / .fastq.gz).")

# ¿Está comprimido? (los ficheros gzip empiezan por los bytes 1f 8b)
with open(origen, "rb") as fh:
    cabecera = fh.read(64)
if cabecera.startswith(b"version https://git-lfs"):
    raise SystemExit("❌ El FASTQ está guardado con Git LFS y no se ha descargado. Súbelo sin LFS (máx. 100 MB) o comprímelo.")
FASTQ = f"{MUESTRA}.fastq.gz" if cabecera[:2] == b"\x1f\x8b" else f"{MUESTRA}.fastq"
for viejo in (f"{MUESTRA}.fastq", f"{MUESTRA}.fastq.gz"):   # quita accesos directos anteriores
    if os.path.islink(viejo): os.remove(viejo)
os.symlink(origen, FASTQ)          # acceso directo: no duplica el fichero
print("FASTQ:", origen, f"({os.path.getsize(origen)/1e6:.1f} MB)")

# ¿Trae las señales de metilación de Dorado (etiquetas MM/ML en la cabecera de cada lectura)?
import gzip as _gz
with (_gz.open(FASTQ, "rt") if cabecera[:2] == b"\x1f\x8b" else open(FASTQ)) as fh:
    METILACION = "MM:Z:" in fh.readline()
print("Metilación (MM/ML):", "✅ sí" if METILACION else "❌ no (se saltará el apartado de metilación)")

# 2) Referencia: un cromosoma del hg38
REF = f"{CROMOSOMA}.fa"
if not os.path.exists(REF + ".fai"):
    ejecuta(f"wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/{CROMOSOMA}.fa.gz && gunzip -f {CROMOSOMA}.fa.gz && samtools faidx {REF}", [
        ("wget", "descarga el cromosoma desde UCSC"),
        ("gunzip", "lo descomprime"),
        ("samtools faidx", "crea el índice .fai de la referencia")])
ejecuta("ls -lh")

❓ **Pregunta 1.** Abre el FASTQ y mira las 4 primeras líneas. ¿Qué hay en cada una? ¿Qué codifica la cuarta?

💡 Fíjate en la **primera línea**: además del nombre de la lectura aparecen las etiquetas `MM:Z:C+m?,…` (posiciones de las C) y `ML:B:C,…` (probabilidad de que cada una esté metilada, de 0 a 255). ¡La metilación viaja dentro del FASTQ!

In [ ]:
!zcat -f {FASTQ} 2>/dev/null | head -4 | cut -c1-250

## 3. Control de calidad de las lecturas

In [ ]:
import gzip, math, statistics
import matplotlib.pyplot as plt

def leer_fastq(path):
    with open(path, "rb") as fh:
        es_gz = fh.read(2) == b"\x1f\x8b"
    op = gzip.open if es_gz else open
    with op(path, "rt") as f:
        while True:
            h = f.readline()
            if not h: break
            s = f.readline().strip(); f.readline(); q = f.readline().strip()
            yield s, q

longitudes, calidades = [], []
for s, q in leer_fastq(FASTQ):
    longitudes.append(len(s))
    # Q media correcta: se promedian probabilidades de error, no valores Phred
    p = sum(10 ** (-(ord(c) - 33) / 10) for c in q) / len(q)
    calidades.append(-10 * math.log10(p))

orden = sorted(longitudes, reverse=True); total = sum(orden); acum = 0
for L in orden:
    acum += L
    if acum >= total / 2: n50 = L; break

print(f"Lecturas:            {len(longitudes):,}")
print(f"Bases totales:       {total/1e6:,.1f} Mb")
print(f"Longitud media:      {statistics.mean(longitudes):,.0f} pb")
print(f"N50:                 {n50:,} pb")
print(f"Lectura más larga:   {max(longitudes):,} pb")
print(f"Q media (mediana):   {statistics.median(calidades):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(longitudes, bins=60, color="#2b6cb0"); ax[0].set_xlabel("Longitud de lectura (pb)"); ax[0].set_ylabel("Nº lecturas")
ax[0].axvline(n50, color="crimson", ls="--", label=f"N50 = {n50:,}"); ax[0].legend()
ax[1].hist(calidades, bins=40, color="#38a169"); ax[1].set_xlabel("Calidad media por lectura (Phred)")
plt.tight_layout(); plt.show()

❓ **Pregunta 2.** ¿Qué es el N50? Compara la longitud de estas lecturas con las de Illumina (2×150 pb). ¿Qué tipo de variantes podrás ver ahora que antes no?

## 4. Alineamiento contra el genoma de referencia (minimap2)
`-ax map-ont` = preset para lecturas de nanoporo. El resultado se ordena e indexa con samtools → **BAM + BAI**.

In [ ]:
BAM = f"{MUESTRA}.bam"
copiar_tags = "-y -Y " if METILACION else ""   # -y copia las etiquetas MM/ML al BAM; -Y conserva la secuencia completa en las alineaciones suplementarias

ejecuta(f"minimap2 -t 2 -ax map-ont {copiar_tags}{REF} {FASTQ} | samtools sort -@2 -o {BAM} -", [
    ("minimap2", "alineador para lecturas largas"),
    ("-t 2", "usa 2 procesadores"),
    ("-a", "la salida es un alineamiento en formato SAM"),
    ("-x map-ont", "parámetros optimizados para nanoporo (ONT)"),
    *([("-y", "copia las etiquetas de metilación MM/ML al BAM"), ("-Y", "no recorta la secuencia de las alineaciones suplementarias (para que MM/ML sigan cuadrando)")] if METILACION else []),
    (REF, "genoma de referencia"),
    (FASTQ, "tus lecturas"),
    ("| samtools sort", "ordena las lecturas por posición en el genoma"),
    ("-o " + BAM, "guarda el resultado en formato binario BAM"),
])
ejecuta(f"samtools index {BAM}", [("samtools index", "crea el índice .bai para poder saltar a cualquier región (lo necesita IGV)")])
ejecuta(f"samtools flagstat {BAM}", [("samtools flagstat", "estadísticas: cuántas lecturas han alineado")])


### 🔍 Mira dentro del BAM
Cada línea es una lectura alineada. Columnas: **1** nombre · **2** FLAG · **3** cromosoma · **4** posición · **5** calidad de mapeo (MAPQ) · **6** CIGAR (`M` coincide, `I` inserción, `D` deleción, `S` soft-clip) · … · **10** secuencia.

In [ ]:
ejecuta(f"samtools view {BAM} | head -3 | cut -f1-6,10 | cut -c1-250", [("samtools view", "convierte el BAM (binario) en texto legible"), ("head -3", "solo las 3 primeras lecturas"), ("cut", "recorta columnas y caracteres para que quepa en pantalla")])

❓ **Pregunta 3.** ¿Qué porcentaje de lecturas ha alineado? ¿Qué significan las lecturas *supplementary*? (pista: piensa en una lectura que cruza una translocación o una inversión)

## 5. Calidad y cobertura en la región de interés (SERPINC1)
Ahora nos centramos en las coordenadas del gen: ¿cuántas lecturas lo cubren, con qué profundidad y con qué calidad de mapeo?

In [ ]:
ejecuta(f"samtools coverage -r {REGION} {BAM}", [("samtools coverage", "profundidad y % cubierto"), ("-r " + REGION, "solo en tu región de interés")])
ejecuta(f"samtools coverage -r {REGION} -m {BAM}", [("-m", "dibuja un histograma de la cobertura")])

In [ ]:
import pysam, numpy as np
crom, rango = REGION.replace(",", "").split(":"); ini, fin = map(int, rango.split("-"))

# Perfil de profundidad base a base en la región
prof = np.zeros(fin - ini + 1, dtype=int)
for linea in subprocess.run(f"samtools depth -a -r {REGION} {BAM}", shell=True, capture_output=True, text=True).stdout.splitlines():
    _, pos, dp = linea.split("\t"); prof[int(pos) - ini] = int(dp)

# Lecturas de la región: longitud, MAPQ y cuántas la cruzan entera
lecturas = [r for r in pysam.AlignmentFile(BAM).fetch(crom, ini, fin) if not (r.is_secondary or r.is_supplementary)]
largos = [r.query_length for r in lecturas]; mapq = [r.mapping_quality for r in lecturas]
completas = sum(1 for r in lecturas if r.reference_start <= ini and r.reference_end >= fin)

print(f"Región {GEN}: {REGION}  ({(fin-ini)/1000:.1f} kb)")
print(f"Lecturas que tocan la región:        {len(lecturas)}")
print(f"Lecturas que la cruzan de lado a lado: {completas}" + ("  ← ¡una sola lectura lee todo el gen!" if completas else "  (la región es más larga que las lecturas)"))
print(f"Profundidad media / mínima:          {prof.mean():.1f}x / {prof.min()}x")
print(f"Bases con ≥10x:                      {100*(prof>=10).mean():.1f} %")
print(f"MAPQ mediana:                        {np.median(mapq):.0f}  (60 = mapeo único)")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5), gridspec_kw={"width_ratios": [3, 1, 1]})
ax[0].fill_between(np.arange(ini, fin + 1), prof, color="#2b6cb0", alpha=.8, step="mid")
ax[0].axhline(prof.mean(), color="crimson", ls="--", label=f"media {prof.mean():.0f}x"); ax[0].legend()
ax[0].set_title(f"Cobertura en {GEN}"); ax[0].set_xlabel(f"{crom} (hg38)"); ax[0].set_ylabel("Profundidad")
ax[0].ticklabel_format(axis="x", style="plain", useOffset=False)
ax[1].hist(largos, bins=30, color="#38a169"); ax[1].set_title("Longitud de las lecturas"); ax[1].set_xlabel("pb")
ax[2].hist(mapq, bins=range(0, 62, 4), color="#dd6b20"); ax[2].set_title("Calidad de mapeo (MAPQ)")
plt.tight_layout(); plt.show()

❓ **Pregunta 4.** ¿Cuál es la profundidad media en el gen? ¿Hay zonas con menos cobertura? ¿Cuántas lecturas cruzan el gen entero? Este experimento usó **adaptive sampling**: compara con la cobertura del resto del cromosoma (celda siguiente).

In [ ]:
!samtools coverage {BAM}

## 6. Llamada de variantes estructurales (Sniffles2)

In [ ]:
VCF = f"{MUESTRA}.sv.vcf"
ejecuta(f"sniffles --input {BAM} --vcf {VCF} --reference {REF} --threads 2 --output-rnames", [
    ("sniffles", "detecta variantes estructurales (>50 pb): DEL, INS, DUP, INV, BND"),
    ("--input " + BAM, "tus lecturas alineadas"),
    ("--vcf " + VCF, "fichero de salida con las variantes"),
    ("--reference " + REF, "permite escribir la secuencia de las inserciones"),
    ("--output-rnames", "apunta qué lecturas apoyan cada variante (lo usaremos para separar alelos)"),
])
ejecuta(f"bgzip -kf {VCF} && tabix -f -p vcf {VCF}.gz", [("bgzip / tabix", "comprime e indexa el VCF para IGV")])
print("\n🔍 Mira dentro del VCF (cabecera + primera variante):")
ejecuta(f"grep -m 6 '^##INFO' {VCF}; grep -v '^##' {VCF} | head -2 | cut -c1-300")


import pandas as pd
filas = []
for linea in open(VCF):
    if linea.startswith("#"): continue
    c = linea.rstrip("\n").split("\t")
    info = dict(kv.split("=", 1) if "=" in kv else (kv, True) for kv in c[7].split(";"))
    filas.append(dict(CROM=c[0], POS=int(c[1]), ID=c[2], TIPO=info.get("SVTYPE"),
                      TAMAÑO=info.get("SVLEN"), LECTURAS=info.get("SUPPORT"),
                      GENOTIPO=c[9].split(":")[0], FILTRO=c[6], ALT=c[4],
                      LECTURAS_ID=info.get("RNAMES", "")))
svs = pd.DataFrame(filas)
print(f"Variantes estructurales detectadas en {CROMOSOMA}: {len(svs)}")
en_region = svs[(svs.CROM == crom) & (svs.POS.between(ini, fin))]
print(f"En la región de {GEN} ({REGION}): {len(en_region)}")
en_region.drop(columns=["ALT", "LECTURAS_ID"])

❓ **Pregunta 5.** ¿Qué tipo de variante encuentras en el gen? ¿De qué tamaño? ¿Está en homocigosis o heterocigosis? ¿Cuántas lecturas la soportan?

### 6b. ¿Qué es la secuencia insertada?
Guardamos la secuencia de las inserciones de la región en un FASTA. Cópiala y pégala en **[Dfam](https://dfam.org/search/sequence)** (base de datos de elementos repetitivos) o en **[BLAST](https://blast.ncbi.nlm.nih.gov/)**.

In [ ]:
ins = en_region[(en_region.TIPO == "INS") & (en_region.ALT.str.len() > 50)]
with open("inserciones.fa", "w") as f:
    for _, r in ins.iterrows():
        f.write(f">{r.ID}_{r.CROM}:{r.POS}_{len(r.ALT)}pb\n{r.ALT}\n")
!cat inserciones.fa | cut -c1-2000

❓ **Pregunta 6.** ¿Qué tipo de elemento es? ¿Cómo ha llegado ahí? (recuerda la diapositiva de retrotransposones: L1, Alu, **SVA**…). ¿Por qué ninguna de las técnicas previas lo detectó?

## 7. Metilación por alelo (Methylartist)
Las lecturas de nanoporo leen el ADN **nativo**, así que además de la secuencia nos dicen si cada CpG está **metilado (5mC)**.

Como las lecturas son largas, podemos saber de qué **alelo** viene cada una. Vamos a separarlas en dos grupos:
* **Alelo 1 (con la variante):** las lecturas que, según Sniffles2, llevan la variante estructural.
* **Alelo 2 (sin la variante):** las lecturas que pasan por ese mismo punto **sin** la variante.

Marcamos cada lectura con la etiqueta `HP` (haplotipo) y dibujamos la metilación de cada alelo con **Methylartist**.

In [ ]:
if not METILACION:
    print("⚠️ Este FASTQ no trae etiquetas de metilación: sáltate este apartado.")
else:
    # 1) Variante principal de la región (la que más lecturas apoyan)
    sv_reg = en_region.copy()
    sv_reg["LECTURAS"] = sv_reg["LECTURAS"].astype(int)
    if len(sv_reg):
        sv = sv_reg.sort_values("LECTURAS", ascending=False).iloc[0]
        apoyan = set(sv.LECTURAS_ID.split(","))
        tam = abs(int(sv.TAMAÑO)) if str(sv.TAMAÑO).lstrip("-").isdigit() else 1
        sv_ini, sv_fin = sv.POS, sv.POS + (tam if sv.TIPO == "DEL" else 1)
        print(f"Variante usada para separar alelos: {sv.TIPO} de {tam} pb en {sv.CROM}:{sv.POS} ({len(apoyan)} lecturas)")
    else:
        apoyan, sv_ini, sv_fin = set(), None, None
        print("No hay variantes estructurales en la región: dibujaremos todas las lecturas juntas.")

    # 2) Etiquetamos cada lectura con su alelo (HP=1 con la variante, HP=2 sin ella)
    BAM_HP = f"{MUESTRA}.alelos.bam"
    n1 = n2 = 0
    with pysam.AlignmentFile(BAM) as ent, pysam.AlignmentFile(BAM_HP, "wb", template=ent) as sal:
        for r in ent.fetch(crom, ini, fin):
            if r.is_secondary or r.is_supplementary:
                continue   # solo la alineación principal de cada lectura
            if apoyan:
                if r.query_name in apoyan:
                    r.set_tag("HP", 1); r.set_tag("PS", int(sv_ini)); n1 += 1
                elif r.reference_start < sv_ini - 500 and r.reference_end > sv_fin + 500:
                    r.set_tag("HP", 2); r.set_tag("PS", int(sv_ini)); n2 += 1
            sal.write(r)
    pysam.index(BAM_HP)
    if apoyan: print(f"Lecturas alelo 1 (con variante): {n1}   ·   alelo 2 (sin variante): {n2}")

### 7b. Genes de la región (anotación GENCODE)
Descargamos solo los genes de nuestra región para que aparezcan dibujados debajo del gráfico (si falla la descarga, el gráfico se hace igual, sin genes).

In [ ]:
GTF = f"genes_{GEN}.gtf.gz"
if not os.path.exists(GTF) or os.path.getsize(GTF) < 100:
    url_gtf = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_46/gencode.v46.basic.annotation.gtf.gz"
    ejecuta(f"wget -qO- {url_gtf} | zcat | awk -F'\\t' '$1==\"{crom}\" && $4<={fin} && $5>={ini}' "
            f"| sort -k1,1 -k4,4n | bgzip > {GTF} && tabix -p gff {GTF}",
            [("wget … | zcat", "descarga la anotación de genes de GENCODE (hg38)"),
             ("awk", "se queda solo con lo que cae en nuestra región"),
             ("bgzip + tabix", "comprime e indexa, como Methylartist necesita")])
TIENE_GTF = os.path.exists(GTF) and os.path.getsize(GTF) > 100
if not TIENE_GTF:
    for f in (GTF, GTF + ".tbi"):
        if os.path.exists(f): os.remove(f)
print("Anotación de genes:", "✅" if TIENE_GTF else "❌ (seguimos sin ella)")

### 7c. Gráfico de metilación
Cómo leer el gráfico de Methylartist:
* **Arriba:** cada línea horizontal es una lectura, agrupadas por alelo. Los puntos son CpG: **rellenos (negros) = metilados**, **huecos (de color) = no metilados**.
* **Centro:** traslada cada CpG de la lectura a su posición en el genoma.
* **Abajo:** fracción de metilación suavizada a lo largo de la región, una línea por alelo. En la leyenda, **`.1` = alelo con la variante** y **`.2` = alelo sin la variante**.
* La **franja sombreada** marca la posición de la variante estructural.

In [ ]:
if METILACION:
    from IPython.display import Image, display
    PNG = f"metilacion_{GEN}.png"
    opciones = [("methylartist locus", "dibuja la metilación de una región"),
                ("-b " + (BAM_HP if apoyan else BAM), "BAM con etiquetas MM/ML" + (" y HP (alelo)" if apoyan else "")),
                ("-i " + REGION, "región a dibujar"),
                ("-r " + REF + " -n CG", "referencia y motivo: metilación en CpG"),]
    cmd = f"methylartist locus -b {BAM_HP if apoyan else BAM} -i {REGION} -r {REF} -n CG --primary_only -o {PNG}"
    if apoyan:
        cmd += f" --phased --ignore_ps -l {sv_ini}-{sv_fin + 50}"
        opciones += [("--phased", "separa las lecturas por alelo (etiqueta HP)"),
                     ("-l", "sombrea la posición de la variante")]
    if TIENE_GTF:
        cmd += f" -g {GTF}"; opciones += [("-g " + GTF, "dibuja los genes")]
    ejecuta(cmd, opciones)
    display(Image(PNG, width=1100))

In [ ]:
if METILACION and apoyan:
    # Metilación media de cada lectura (fracción de CpG con probabilidad ≥ 50 %)
    filas = []
    for r in pysam.AlignmentFile(BAM_HP).fetch(crom, ini, fin):
        if not r.has_tag("HP") or not r.modified_bases: continue
        probs = [p for (base, hebra, mod), v in r.modified_bases.items() if mod == "m" for _, p in v]
        if probs: filas.append(dict(alelo="con variante" if r.get_tag("HP") == 1 else "sin variante",
                                    metilacion=np.mean(np.array(probs) >= 128)))
    met = pd.DataFrame(filas)
    print(met.groupby("alelo").metilacion.describe()[["count", "mean", "50%"]].round(2))
    met.boxplot(column="metilacion", by="alelo", figsize=(5, 4), grid=False)
    plt.suptitle(""); plt.title("Metilación media por lectura"); plt.ylabel("Fracción de CpG metilados"); plt.show()

❓ **Pregunta 7.** ¿Tienen los dos alelos el mismo nivel de metilación? ¿Cambia la metilación cerca de la variante? Si la variante es un retrotransposón (SVA, L1, Alu…), ¿qué esperarías y por qué? (pista: la célula silencia los retrotransposones metilándolos).

## 8. Visualización en IGV (dentro del cuaderno)
Se abre un IGV aquí mismo con tu BAM y tu VCF. Muévete, haz zoom y busca la variante.

💡 Consejos: activa *“Show soft clips”*, colorea las lecturas por *“insertion size”* y fíjate en las **marcas moradas** (inserciones) en las lecturas.

In [ ]:
if not EN_COLAB:
    print("En Codespaces IGV no se abre dentro del cuaderno: pasa directamente al paso 9 y usa igv.org/app.")
else:
  import igv_notebook
  igv_notebook.init()
  b = igv_notebook.Browser({"genome": "hg38", "locus": GEN})
  b.load_track({"name": f"{MUESTRA} SV", "path": f"{TRABAJO}/{VCF}.gz",
                "indexPath": f"{TRABAJO}/{VCF}.gz.tbi", "format": "vcf", "type": "variant"})
  b.load_track({"name": f"{MUESTRA} lecturas", "path": f"{TRABAJO}/{MUESTRA}.alelos.bam" if METILACION and os.path.exists(f"{MUESTRA}.alelos.bam") else f"{TRABAJO}/{BAM}",
                "indexPath": (f"{TRABAJO}/{MUESTRA}.alelos.bam.bai" if METILACION and os.path.exists(f"{MUESTRA}.alelos.bam") else f"{TRABAJO}/{BAM}.bai"), "format": "bam", "type": "alignment",
                "showSoftClips": True, **({"colorBy": "basemod2"} if METILACION else {})})

## 9. Descarga tus resultados y ábrelos en IGV
1. Ejecuta la celda: se descarga `resultados_<muestra>.zip`.
2. Abre **IGV Desktop** (o sin instalar nada: **[igv.org/app](https://igv.org/app)**), genoma **hg38**.
3. `File → Load from File…` y selecciona el `.bam` (con su `.bai` en la misma carpeta) y el `.vcf.gz`.
4. Busca el gen y haz una **captura de pantalla** de la variante: ¡es tu informe!
5. **Metilación en IGV:** carga `…alelos.bam`, botón derecho sobre las lecturas → **Color alignments by → base modification 2-color (5mC)** y **Group alignments by → phase** (o *tag HP*). Verás los dos alelos separados, con los CpG metilados en rojo y los no metilados en azul.

In [ ]:
!zip -q resultados_{MUESTRA}.zip {BAM} {BAM}.bai {VCF}.gz {VCF}.gz.tbi inserciones.fa {MUESTRA}.alelos.bam* metilacion_*.png 2>/dev/null
if EN_COLAB:
    from google.colab import files
    files.download(f"resultados_{MUESTRA}.zip")
else:
    print(f"Tu fichero está en: {TRABAJO}/resultados_{MUESTRA}.zip")
    print("Descárgalo: panel de la izquierda (Explorador) → carpeta 'resultados' → botón derecho sobre el .zip → Download")

## 🏁 Cierre
Comparte en el chat de clase:
1. Gen y posición de la variante · 2. Tipo y tamaño · 3. Qué elemento es · 4. ¿Está metilado? ¿Igual en los dos alelos? · 5. Tu captura de IGV.

**Para pensar:** ¿Cómo validarías este hallazgo en el laboratorio? ¿Cómo lo describirías en nomenclatura HGVS?

---
*Extensión (si te sobra tiempo):* cambia `REGION` a otra zona del cromosoma y busca otras variantes estructurales. ¿Cuántas son polimorfismos comunes? (consulta gnomAD-SV).